# Libraries

In [ ]:
import pandas as pd
from pathlib import Path
import os
from dotenv import load_dotenv

# 1. Data loading

In [ ]:
# BMS: Setting up on my local machine for now
load_dotenv()
BASE = Path(os.environ["BATTERY_DATA_ROOT"])

In [ ]:
folders = {
    'dnsp_10': (BASE / os.environ['DNSP_10_DIR'], range(26, 51)),
    'dnsp_9': (BASE / os.environ['DNSP_9_DIR'], range(1, 26)),
    'dnsp_5': (BASE / os.environ['DNSP_5_DIR'], range(51, 101)),
}

In [ ]:
# Aggregating all files into a single DataFrame
dfs = []
for region, (folder, sn_range) in folders.items():
    for sn in sn_range:
        sn_id = f'SN{sn:03d}'
        # prefer parquet, fall back to csv
        parquet_path = folder / f'{sn_id}_telemetry_60sec.parquet'
        csv_path     = folder / f'{sn_id}_telemetry_60sec.csv'
        path = parquet_path if parquet_path.exists() else csv_path
        
        try:
            df = pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
            df['region'] = region
            dfs.append(df)
        except FileNotFoundError:
            print(f'Missing: {path}')

all_data = pd.concat(dfs, ignore_index=True)

# Fix types once, up front
all_data['timestamp'] = pd.to_datetime(all_data['timestamp'])
all_data['pseudonym'] = all_data['pseudonym'].astype('category')  # saves memory
all_data['region']    = all_data['region'].astype('category')

# save the combined file as parquet for fast re-loading next session
all_data.to_parquet(BASE / 'combined_telemetry.parquet', index=False)

In [ ]:
all_data